# N04: hybrid, and the full dev ladder

R5 adds the text branch (R3) and the image branch (R4). Their raw cosine similarities live on different scales, so
each is z-scored per query first, then averaged with equal weight (alpha = 0.5). Alpha is **not tuned**: tuning it on
judgments pooled from a few alphas would favour those alphas (ADR 0007).

This notebook ends the dev phase: full table, the 3 pre-registered comparisons, then `freeze()`.

In [ ]:
from pathlib import Path

import pandas as pd

from deep_learning_project_waerniers_lamy.nlp import utils as u

In [ ]:
paintings = u.load_corpus("dev")
descriptions = u.load_descriptions()
queries = u.QUERIES["text"].tolist()
scores = {
    "R0 random": u.rung_random(paintings, queries),
    "R1 metadata": u.rung_metadata(paintings, queries),
    "R2 description": u.rung_description(paintings, queries, descriptions),
    "R3 description+metadata": u.rung_description_metadata(paintings, queries, descriptions),
    "R4 siglip image": u.rung_siglip(paintings, queries),
    "R5 hybrid": u.rung_hybrid(paintings, queries, descriptions),
}
u.judge_with_llm("dev", scores, paintings)

### Judging

Every painting in the new top 5 of a rung is graded by an LLM judge (`u.JUDGE_MODEL`, another model family than the
description generator). It sees **only the query and the image**, answers two yes/no questions (is the main subject
there? does it match the whole query?) and the grade follows by rule: 0, 1 or 2. Grades are saved in
`results/nlp/qrels_dev.json`; already graded pairs are never sent again. The judge is validated against 150 of your own
grades in N04 before any result is frozen.

## 0. Validate the judge (by hand, about 10 minutes)

A seeded random sample of 150 graded (query, painting) pairs, drawn from every rung. Open the page, grade each pair with
the keyboard on the same scale (0 subject absent, 1 subject there but part of the query missing, 2 whole match), click
*export JSON*, then run the next cells. The judge is accepted if the weighted kappa with your grades is at least
0.6 ("substantial agreement"); otherwise `freeze()` refuses and dev must be graded by hand.

In [ ]:
u.write_calibration_page("dev", paintings)

In [ ]:
u.import_calibration("dev", Path.home() / "Downloads" / "calibration_dev.json")
u.judge_agreement("dev")

## 1. The dev ladder

In [ ]:
per_query = u.per_query_scores("dev", scores, paintings)
table = u.summary_table(per_query)
u.plot_ladder(table, "dev: full ladder")
table.to_csv(u.RESULTS_DIR / "dev_summary.csv", index=False)
per_query.to_csv(u.RESULTS_DIR / "dev_per_query.csv", index=False)
table

## 2. Pre-registered comparisons

Only these three, fixed before seeing results, Holm-corrected: visual text vs metadata, visual embedding vs metadata,
and fusion vs the better single branch (R3 or R4, whichever is higher on dev; that choice is frozen for Museum B).
With 40 queries, differences of a few nDCG points are not detectable: read the confidence intervals, not only p.

In [ ]:
means = table.set_index("rung")["nDCG@5"]
best_single = max(["R3 description+metadata", "R4 siglip image"], key=means.get)
u.compare(per_query, [("R2 description", "R1 metadata"), ("R4 siglip image", "R1 metadata"), ("R5 hybrid", best_single)])

## 3. By query category (descriptive, no p-values: about 10 queries per category)

In [ ]:
per_query.groupby(["rung", "category"])["ndcg@5"].mean().unstack().round(3)

## 4. Limits of the measurement

- **Pool miss rate**: R0's random paintings that no real rung put in its top 5 were judged too. The share of them
  that is relevant estimates how many relevant paintings the pool never saw (they count as 0 for everyone).
- **Latency** on this CPU, for one query, corpus embeddings already computed.
- The grades come from an LLM judge: read them with the kappa of section 0 (agreement with a human, not truth).

In [ ]:
qrels = u.load_qrels("dev")
real = [s for name, s in scores.items() if name != "R0 random"]
outside = [qrels[qid][pid]
           for qi, qid in enumerate(u.QUERIES["query_id"])
           for pid in u.top_k_ids(scores["R0 random"], paintings)[qi]
           if all(pid not in u.top_k_ids(s, paintings)[qi] for s in real) and pid in qrels.get(qid, {})]
print(f"pool miss rate: {sum(g >= 1 for g in outside) / len(outside):.3f} over {len(outside)} random paintings")

rungs = {
    "R1 metadata": (u.rung_metadata, paintings),
    "R2 description": (u.rung_description, paintings, descriptions),
    "R4 siglip image": (u.rung_siglip, paintings),
    "R5 hybrid": (u.rung_hybrid, paintings, descriptions),
}
pd.Series({name: u.latency_ms(fn, *args) for name, (fn, *args) in rungs.items()}, name="median ms / query").round(1)

## 5. Freeze

Writes `results/nlp/frozen_config.json`: models, prompts, judge, alpha, the judge's kappa, and hashes of the
descriptions and the queries. Refuses if the judge kappa is below 0.6. N05
refuses to run if any of them changed afterwards. Commit the frozen config with the dev judgments before opening N05.

In [ ]:
u.freeze(best_single)
print(Path(u.FROZEN_CONFIG).read_text())